# 4. Parser performance

*Author: La Anh Hào · code: `src/15_run_parsing.ipynb`*

The statement asks for the running time of the parser (end-to-end wall time and average time per paper) and
for its memory footprint (maximum RAM and the size of the final output), measured on Google Colab in CPU-only
mode. `src/15_run_parsing.ipynb` measures them itself while it parses: every session writes a record with its
wall time, the papers it parsed and its RAM, sampled every 2 seconds for the notebook process **and all its
worker processes**; the record is saved every few minutes, so a session that Colab stops still counts.
`m2_benchmark.json` sums the sessions.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import report_data as rd

rd.check()
bench, stats = rd.benchmark(), rd.stats()
sessions, versions, papers = rd.sessions(), rd.versions(), rd.papers()
main = sessions.loc[sessions["papers"].idxmax()]   # the session that parsed the most papers
parsers = sessions["parser_version"].str.split("|").str[0]
first = sessions[parsers == parsers.iloc[0]]        # the sessions of the first parser version
full_pass = int(first["papers"].sum()) >= int(stats["papers"])
later = sessions[parsers != parsers.iloc[0]]
pass_wall = first["wall_seconds"].sum() if full_pass else bench["wall_seconds"]
hm = rd.hm

## 4.1 Testbed and sessions

`15_run_parsing` runs `parse_paper` (chapter [2](hierarchy.md)) in a pool of worker processes, one paper per
task, on a Colab CPU runtime with the corpus on Google Drive. Reading the source files from Drive, not the
CPU, limits the speed, so the pool has more workers than the runtime has cores. A session parses the papers
that the resume index does not list yet and builds the three Parquet tables at the end.

In [ ]:
t41 = pd.DataFrame({
    "Start (UTC)": sessions["start"].dt.strftime("%Y-%m-%d %H:%M"),
    "Wall time": [hm(s) for s in sessions["wall_seconds"]],
    "Papers parsed": sessions["papers"],
    "Versions": sessions["versions"],
    "Workers": sessions["workers"],
    "Peak RAM (MB)": sessions["ram_peak_mb"],
    "Parser": sessions["parser_version"].str.split("|").str[0],
    "Status": np.where(sessions["complete"], "completed", "interrupted"),
})
rd.table(t41, "Table 4.1 – Sessions of 15_run_parsing", {"Peak RAM (MB)": "{:,.0f}"})
text = (f"Runtime: `{bench.get('colab_cpu', '?')}`, **{bench.get('workers', '?')}** worker processes. "
        f"The main session ({main['start']:%d %b}) parsed **{main['papers']:,}** papers in "
        f"**{hm(main['wall_seconds'])}**")
if full_pass and len(first) > 1:
    n_before = len(first) - 1
    text += (f"; with the {'session' if n_before == 1 else f'{n_before} sessions'} before it, parser "
             f"`{parsers.iloc[0]}` parsed all "
             f"**{int(first['papers'].sum()):,}** papers in **{hm(pass_wall)}**")
text += "."
if len(later):
    by_p = later.assign(p=parsers.loc[later.index]).groupby("p", sort=False).agg(
        n_sess=("papers", "size"), papers=("papers", "sum"), wall=("wall_seconds", "sum")
    )
    parts = [f"`{p}` ({int(r.n_sess)} {'session' if r.n_sess == 1 else 'sessions'}, {int(r.papers):,} papers, {hm(r.wall)})"
             for p, r in by_p.iterrows()]
    text += (f" Targeted carry-over updates then ran on the flagged papers and verification samples only: "
             + "; ".join(parts) + " (chapter [2.5](running.md)).")
rd.md(text)

## 4.2 Running time

**End-to-end wall time** is the time between the start and the end of the sessions, setup and the
construction of the tables included. The **average time per paper** divides it by the number of papers: it is
the throughput of the whole pipeline, with its parallel workers. The time a worker spends on one paper
(reading its files, parsing every version, writing `data/parsed/<id>.json`) is recorded too; it is longer
than the average because several workers run at once.

In [ ]:
worker = papers["seconds"].dropna()
worker = worker[worker > 0]
rows = []
if full_pass:
    rows += [(f"End-to-end wall time of one full pass (parser {parsers.iloc[0]})",
              f"{hm(pass_wall)} for {int(first['papers'].sum()):,} papers"),
             ("Average time per paper, full pass", f"{pass_wall / int(first['papers'].sum()):.3f} s")]
if len(later):
    for p, grp in later.assign(p=parsers.loc[later.index]).groupby("p", sort=False):
        lbl = f"Carry-over update to parser {p} ({len(grp)} {'session' if len(grp) == 1 else 'sessions'}, tables rebuilt)"
        rows.append((lbl, f"{hm(grp['wall_seconds'].sum())} for {int(grp['papers'].sum()):,} papers"))
rows += [
    ("End-to-end wall time, all sessions", hm(bench["wall_seconds"])),
    (f"Average time per paper, all sessions ({hm(bench['wall_seconds'])} / {bench['papers']:,} papers)",
     f"{bench['avg_seconds_per_paper']:.3f} s"),
    ("Time per paper in a worker: median / 90th percentile / maximum",
     f"{worker.median():.2f} s / {worker.quantile(0.9):.2f} s / {worker.max():,.1f} s"),
]
rd.table(pd.DataFrame(rows, columns=["Metric", "Value"]), "Table 4.2 – Running time")

In [ ]:
bins = np.logspace(np.log10(worker.min()), np.log10(worker.max()) + 0.02, 60)
h_w, edges = np.histogram(worker, bins=bins)
centers = np.sqrt(edges[:-1] * edges[1:])
sized = papers[(papers["seconds"] > 0) & (papers["nodes"] > 0)]
sample = sized.sample(min(len(sized), 5000), random_state=0)

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("Time per paper in a worker", f"Time vs tree size ({len(sample):,}-paper sample)"),
)
fig.add_trace(
    go.Bar(
        x=centers, y=h_w, marker_color=rd.BLUE, showlegend=False,
        customdata=np.stack([edges[:-1], edges[1:]], axis=-1),
        hovertemplate="%{customdata[0]:.2f}–%{customdata[1]:.2f} s<br><b>%{y:,}</b> papers<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_vline(
    x=float(worker.median()), line_color=rd.ORANGE, line_width=1.8, line_dash="dash",
    annotation_text=f"median {worker.median():.2f} s", annotation_position="top right",
    row=1, col=1,
)
fig.add_trace(
    go.Scattergl(
        x=sample["nodes"], y=sample["seconds"], mode="markers",
        marker=dict(size=4, color=rd.BLUE, opacity=0.32), showlegend=False,
        customdata=np.stack([sample["arxiv_id"], sample["versions"]], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]} v)<br>Nodes: %{x:,}<br>Time: %{y:.2f} s<extra></extra>",
    ),
    row=1, col=2,
)
fig.update_layout(title="Worker parsing time per paper")
fig.update_xaxes(type="log", title_text="Seconds in a worker (log scale)", row=1, col=1)
fig.update_yaxes(title_text="Papers", row=1, col=1)
fig.update_xaxes(type="log", title_text="Tree nodes of the paper, all versions (log)", row=1, col=2)
fig.update_yaxes(type="log", title_text="Seconds in a worker (log)", row=1, col=2)
rd.show(fig, height=370)
rd.figure_caption("Figure 4.1 – Left: time a worker spent on each paper (orange dashed line: median). "
                  "Right: the same time against the number of tree nodes of the paper.")
slow = papers.nlargest(8, "seconds")[["arxiv_id", "versions", "nodes", "bib_before", "seconds"]]
rd.table(slow.rename(columns={"arxiv_id": "Paper", "versions": "Versions", "nodes": "Tree nodes",
                              "bib_before": "BibTeX entries", "seconds": "Seconds"}),
         "Table 4.3 – The slowest papers", {"Seconds": "{:,.1f}"})

In [ ]:
rho = sized["nodes"].rank().corr(sized["seconds"].rank())   # Spearman: Pearson of the ranks
timeouts = int(versions["error"].str.startswith("timeout").sum())
rd.md(f"The time grows with the size of the paper (Spearman correlation between tree nodes and seconds: "
      f"**{rho:.2f}**). `15` stops a paper after `PAPER_TIMEOUT` = 600 s and records its versions as errors "
      f"instead of blocking the run; "
      + ("no paper reached that limit." if timeouts == 0 else f"{timeouts:,} versions reached that limit."))

## 4.3 Memory footprint

In [ ]:
series = pd.DataFrame(bench.get("ram_timeseries", []), columns=["seconds", "mb"])
if len(series):
    gap = series["seconds"].diff().fillna(0)
    counts = sessions["ram_samples"] if "ram_samples" in sessions.columns else None
    if counts is not None and counts.notna().all() and int(counts.sum()) == len(series):
        starts = [0] + list(np.cumsum(counts.astype(int)))          # the samples of each session, in order
    else:   # older logs: a pause of more than 10 min without a sample starts a new session
        starts = [0] + list(np.where(gap > 600)[0]) + [len(series)]
    pause = pd.Series(0.0, index=series.index)
    for a in starts[1:-1]:
        pause.iloc[a] = gap.iloc[a] - 2          # the time between two sessions, removed from the axis
    series["hours"] = (series["seconds"] - pause.cumsum()) / 3600

    fig = go.Figure()
    palette = [rd.BLUE, rd.BLUE, rd.AQUA, rd.ORANGE, rd.ORANGE, rd.ORANGE]
    for s_idx, (a, b) in enumerate(zip(starts[:-1], starts[1:])):
        part = series.iloc[a:b]
        if not len(part):
            continue
        p_ver = parsers.iloc[s_idx] if s_idx < len(parsers) else "?"
        step = max(1, len(part) // 400)
        sub = part.iloc[::step]
        fig.add_trace(go.Scatter(
            x=sub["hours"], y=sub["mb"], mode="lines",
            name=f"S{s_idx + 1} ({p_ver})",
            line=dict(color=palette[s_idx % len(palette)], width=1.6),
            hovertemplate=f"Session {s_idx + 1} (parser {p_ver})<br>Hour %{{x:.2f}}<br>"
                          "RAM: <b>%{y:,.0f} MB</b><extra></extra>",
        ))
        if a:
            fig.add_vline(x=float(part["hours"].iloc[0]), line_color=rd.AXIS, line_width=1, line_dash="dash")
    fig.add_hline(
        y=bench["peak_ram_mb"], line_color=rd.ORANGE, line_width=1.2, line_dash="dot",
        annotation_text=f"peak {bench['peak_ram_mb']:,.0f} MB", annotation_position="top right",
    )
    fig.update_layout(
        title="RAM during the run (notebook + worker processes)",
        legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
    )
    fig.update_xaxes(title_text="Hours of parsing (sessions concatenated; dashed line = new session)")
    fig.update_yaxes(title_text="RAM of the parser (MB)", range=[0, bench["peak_ram_mb"] * 1.15])
    rd.show(fig, height=370, margin=dict(l=60, r=25, t=55, b=80))
    rd.figure_caption("Figure 4.2 – Resident memory of the notebook process and its workers, sampled every 2 s.")
rd.md(f"**Peak RAM: {bench['peak_ram_mb']:,.0f} MB**, time-weighted average **{bench['avg_ram_mb']:,.0f} MB**, "
      f"for the notebook and its {bench.get('workers', '?')} workers together. The value is the sum of the resident "
      f"memory of the processes: the workers are forked from the notebook and share some of its memory pages, which "
      f"the sum counts once per process, so it is an upper bound of the memory used. Memory stays steady around "
      f"~2 GB during parsing and peaks during final Parquet table assembly on the 27 Nov session, remaining well "
      f"within a free Colab CPU runtime (~12 GB).")

## 4.4 Output size

The output has two parts: `data/parsed/`, one JSON file per paper with the trees of all its versions and its
rows (the input of Milestone 3, enough to rebuild the tables without parsing again), and the three Parquet
tables of the statement in `data/m2/`.

In [ ]:
last = sessions.iloc[-1]
total = bench["output_bytes"]
parquet = last["parquet_bytes"] if "parquet_bytes" in sessions.columns and pd.notna(last["parquet_bytes"]) else None
parsed_n = int(stats["papers_parsed"])
rows = [("data/parsed/ (one JSON file per paper)", None if parquet is None else (total - parquet) / rd.MB),
        ("data/m2/*.parquet (elements, hierarchy, bibtex)", None if parquet is None else parquet / rd.MB),
        ("Total", total / rd.MB)]
t44 = pd.DataFrame(rows, columns=["Output", "Size (MB)"])
rd.table(t44, "Table 4.4 – Size of the output", {"Size (MB)": "{:,.1f}"})

# Derive final table row counts directly from m2_stats.json (parser 13.9):
# every parsed version has one root document node with no parent, so hierarchy rows = nodes - ok versions.
n_elements = int(stats["elements_after_dedup"])
n_hierarchy = int(stats["elements_before_dedup"]) - int(stats["status"].get("ok", 0))
n_bibtex = int(stats["bib_entries_after_dedup"])
t45 = pd.DataFrame([("elements", n_elements), ("hierarchy", n_hierarchy), ("bibtex", n_bibtex)],
                   columns=["Table", "Rows"])
rd.table(t45, f"Table 4.5 – Rows of the three tables (parser {stats['parser_version'].split('|')[0]})")
rd.md(f"The output takes **{total / rd.GB:,.2f} GB**, **{total / max(parsed_n, 1) / 1024:,.0f} KB** per paper on "
      f"average. Most of it is `data/parsed/`, which repeats the rows with the tree of every version; the Parquet "
      f"tables ({parquet / rd.MB:,.1f} MB) are compressed with zstd and store the element `kind` and `entry_type` "
      f"as dictionary columns.")

## 4.5 Summary of the performance metrics

In [ ]:
pass_papers = int(first["papers"].sum()) if full_pass else bench["papers"]
rows = [
    ("Running time", "End-to-end wall time" + (" (one full pass, 13.7)" if full_pass and len(later) else ""), hm(pass_wall)),
    ("Running time", "End-to-end wall time (all 6 sessions incl. 13.8 & 13.9 updates)", hm(bench["wall_seconds"])),
    ("Running time", "Average time per paper (full pass / all sessions)",
     f"{pass_wall / pass_papers:.3f} s / {bench['avg_seconds_per_paper']:.3f} s"),
    ("Running time", "Papers / versions parsed", f"{bench['papers']:,} / {bench['versions']:,}"),
    ("Memory", "Maximum RAM", f"{bench['peak_ram_mb']:,.0f} MB"),
    ("Memory", "Average RAM", f"{bench['avg_ram_mb']:,.0f} MB"),
    ("Memory", "Final output size", f"{bench['output_bytes'] / rd.GB:,.2f} GB"),
    ("Testbed", "Runtime", f"Google Colab, CPU only: {bench.get('colab_cpu', '?')}"),
    ("Testbed", "Worker processes", f"{bench.get('workers', '?')}"),
]
rd.table(pd.DataFrame(rows, columns=["Group", "Metric", "Value"]), "Table 4.6 – Performance metrics required by the statement")